##  Étape 0 _ Imports et configuration
**Ce qu'on fait :** Charger toutes les bibliothèques nécessaires et fixer le seed pour la reproductibilité.

In [99]:
# ── Bibliothèques standard ──────────────────────────────────────
import pandas as pd
import os

##  Étape 1 _ Chargement des datasets
**Ce qu'on fait :** Lire les 4 fichiers CSV et afficher leurs dimensions et colonnes.  
**Vérification :** On vérifie que chaque dataset est bien chargé et que la colonne cible existe.

In [100]:
# ── Chemins des fichiers ─────────────────────────────────────────
PATHS = {
    'bank' : 'Bank.csv',
    'news' : 'News.xlsx',
    'telecom' : 'Telecom.csv',
    'insurance' : 'Insurance.csv'
}

# ── Chargement ───────────────────────────────────────────────────
raw = {}
for name, path in PATHS.items():
    if os.path.exists(path):
        if path.endswith('.xlsx'):
            raw[name] = pd.read_excel(path)
        else :
            raw[name] = pd.read_csv(path)
        print(f" {name} -> {raw[name].shape[0]} lignes x {raw[name].shape[1]} colonnes")
    else :
        print(f" {name} -> fichier introuvable : {path}")

print()
# ── Aperçu rapide ────────────────────────────────────────────────
for name, df in raw.items():
    print(f"{'_'*190}")
    print(f"{name.upper()}_Colonnes :")
    print(list(df.columns))
    print(f" Types : {dict(df.dtypes.value_counts())}")
    print(f" Manquants : {df.isnull().sum().sum()} valeurs")

 bank -> 10000 lignes x 14 colonnes
 news -> 15855 lignes x 19 colonnes
 telecom -> 7043 lignes x 38 colonnes
 insurance -> 33908 lignes x 17 colonnes

______________________________________________________________________________________________________________________________________________________________________________________________
BANK_Colonnes :
['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']
 Types : {dtype('int64'): np.int64(9), dtype('O'): np.int64(3), dtype('float64'): np.int64(2)}
 Manquants : 0 valeurs
______________________________________________________________________________________________________________________________________________________________________________________________
NEWS_Colonnes :
['SubscriptionID', 'HH Income', 'Home Ownership', 'Ethnicity', 'dummy for Children', 'Year Of Residence', 'Age range', 'Language', 'Ad

##  Étape 2 _ Nettoyage des données
**Ce qu'on fait :** Pour chaque dataset :
1. Identifier et supprimer les colonnes inutiles (IDs, colonnes à fuites de données)
2. Supprimer les doublons
3. Gérer les valeurs manquantes
4. Unifier les expressions équivalentes
5. Créer la colonne cible binaire (0/1)

**Vérification :** On affiche la shape avant/après et le taux de churn.

In [101]:
# ════════════════════════════════════════════════════════════════
# NETTOYAGE BANK
# ════════════════════════════════════════════════════════════════
print("=" * 61)
print("NETTOYAGE DATASET BANK")
print("=" * 61)

df_bank = raw['bank'].copy()
print(f"Shape initiale : {df_bank.shape}") # Avant de nettoyage

# ── 1. Renommer la cible ──────────────────────────────────────────
df_bank.rename(columns={'Exited': 'Churn'}, inplace=True)

# ── 2. Supprimer colonnes inutiles ────────────────────────────────
cols_drop_bank = ['RowNumber', 'CustomerId', 'Surname']
df_bank.drop(columns=cols_drop_bank, inplace=True)
print(f"Après suppression colonnes inutiles : {df_bank.shape}")

# ── 3. Valeurs manquantes ─────────────────────────────────────────
df_bank.dropna(inplace=True)
df_bank.drop_duplicates(inplace=True)

print(f"Distribution cible : {dict(df_bank['Churn'].value_counts())}") # 1  et 0
print(f"Taux de churn : {df_bank['Churn'].mean()*100:.2f}%") 
print(f"Shape finale Bank : {df_bank.shape}")

NETTOYAGE DATASET BANK
Shape initiale : (10000, 14)
Après suppression colonnes inutiles : (10000, 11)
Distribution cible : {0: np.int64(7963), 1: np.int64(2037)}
Taux de churn : 20.37%
Shape finale Bank : (10000, 11)


In [102]:
# ════════════════════════════════════════════════════════════════
# NETTOYAGE NEWS
# ════════════════════════════════════════════════════════════════
print("=" * 100)
print("NETTOYAGE DATASET NEWS")
print("=" * 100)

df_news = raw['news'].copy()
print(f"Shape initiale : {df_news.shape}")
print(f"Colonnes : {list(df_news .columns)}")

# ── Identifier la colonne cible ──────────────────────────────────
# Selon l'article : colonne 'Subscriber'
df_news.rename(columns={'Subscriber': 'Churn'}, inplace=True)

# ── Supprimer colonnes géographiques et peu utiles ───────────────
cols_drop_news = ['Address','State','City','Country',
                  'Zip Code','Nielsen Prizm','Source Channel',
                  'Ethnicity','dummy for Children']
df_news.drop(columns=[c for c in cols_drop_news if c in df_news.columns], inplace=True)

# ── Nettoyage ────────────────────────────────────────────────────
df_news.dropna(inplace=True)
df_news.drop_duplicates(inplace=True)

print(f"Distribution cible : {dict(df_news['Churn'].value_counts())}")
# ── Convertir la cible en nombres ──────────────────────────────────
df_news['Churn'] = df_news['Churn'].map({'NO': 0, 'YES': 1})
print(f"Taux de churn : {df_news['Churn'].mean()*100:.2f}%")
print(f"Shape finale News : {df_news.shape}")

NETTOYAGE DATASET NEWS
Shape initiale : (15855, 19)
Colonnes : ['SubscriptionID', 'HH Income', 'Home Ownership', 'Ethnicity', 'dummy for Children', 'Year Of Residence', 'Age range', 'Language', 'Address', 'State', 'City', 'County', 'Zip Code', 'weekly fee', 'Deliveryperiod', 'Nielsen Prizm', 'reward program', 'Source Channel', 'Subscriber']
Distribution cible : {'NO': np.int64(11752), 'YES': np.int64(2827)}
Taux de churn : 19.39%
Shape finale News : (14579, 11)


In [103]:
# ════════════════════════════════════════════════════════════════
# NETTOYAGE TELECOM
# ════════════════════════════════════════════════════════════════
print("=" * 55)
print("NETTOYAGE DATASET TELECOM")
print("=" * 55)

df_tel = raw['telecom'].copy()
print(f"Shape initiale : {df_tel.shape}")

# ── 1. Supprimer les lignes 'Joined' (pas churn ni stayed) ───────
df_tel = df_tel[df_tel['Customer Status'] != 'Joined'] #Nouveaux arrivés n'ont pas d'historique de churn
print(f"Après suppression 'Joined' : {df_tel.shape}")

# ── 2. Créer la variable cible binaire ──────────────────────────
df_tel['Churn'] = (df_tel['Customer Status'] == 'Churned').astype(int) # 1 = churned et 0 = Stayed
print(f"Distribution cible : {dict(df_tel['Churn'].value_counts())}")
print(f"Taux de churn : {df_tel['Churn'].mean()*100:.2f}%")

# ── 3. Supprimer colonnes non pertinentes ────────────────────────
cols_drop_tel = [
    'Customer ID',        # identifiant unique -> pas prédictif
    'Customer Status',    # dérivée de la cible ->data leakage
    'Churn Category',     # info post-churn -> data leakage
    'Churn Reason',       # info post-churn -> data leakage
    'City',               # trop de modalités
    'Zip Code',           # géographique redondant
    'Latitude',           # géographique
    'Longitude',          # géographique
]
df_tel.drop(columns=[c for c in cols_drop_tel if c in df_tel.columns], inplace=True)
print(f"Après suppression colonnes inutiles : {df_tel.shape}")

# ── 4. Gérer les valeurs manquantes ──────────────────────────────
print(f"Valeurs manquantes avant : {df_tel.isnull().sum().sum()}")
# Colonnes numériques -> médiane
for col in df_tel.select_dtypes(include='number').columns:
    df_tel[col].fillna(df_tel[col].median())
# Colonnes texte -> 'Unknown'
for c in df_tel.select_dtypes(include='object').columns:
    df_tel[col].fillna('Unknown')
print(f"Valeurs manquantes après  : {df_tel.isnull().sum().sum()}")

# ── 5. Supprimer les doublons ─────────────────────────────────────
before = len(df_tel)
df_tel.drop_duplicates(inplace=True)
print(f"Doublons supprimés : {before - len(df_tel)}")
print(f"Shape finale Telecom : {df_tel.shape}")


NETTOYAGE DATASET TELECOM
Shape initiale : (7043, 38)
Après suppression 'Joined' : (6589, 38)
Distribution cible : {0: np.int64(4720), 1: np.int64(1869)}
Taux de churn : 28.37%
Après suppression colonnes inutiles : (6589, 31)
Valeurs manquantes avant : 18326
Valeurs manquantes après  : 18326
Doublons supprimés : 0
Shape finale Telecom : (6589, 31)


In [104]:
# ════════════════════════════════════════════════════════════════
# NETTOYAGE INSURANCE
# ════════════════════════════════════════════════════════════════
print("=" * 55)
print("NETTOYAGE DATASET INSURANCE")
print("=" * 55)

df_ins = raw['insurance'].copy()
print(f"Shape initiale : {df_ins.shape}")
print(f"Colonnes : {list(df_ins.columns)}")

# ── Identifier la colonne cible ──────────────────────────────────
# Dernière colonne = feature_15 (selon l'article)
target_col = df_ins.columns[-1] # Récupère le nom de la dernière colonne
print(f"colonne cible détectée : '{target_col}'")
df_ins.rename(columns={target_col: 'Churn'}, inplace=True)

# ── Nettoyage ────────────────────────────────────────────────────
df_ins.dropna(inplace=True)
df_ins.drop_duplicates(inplace=True)

# ── Vérification ─────────────────────────────────────────────────
print(f"Distribution cible : {dict(df_ins['Churn'].value_counts())}")
print(f"Taux de churn : {df_ins['Churn'].mean()*100:.2f}%")
print(f"Shape finale Insurance : {df_ins.shape}")

NETTOYAGE DATASET INSURANCE
Shape initiale : (33908, 17)
Colonnes : ['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15', 'labels']
colonne cible détectée : 'labels'
Distribution cible : {0: np.int64(29941), 1: np.int64(3967)}
Taux de churn : 11.70%
Shape finale Insurance : (33908, 17)
